# v01 — YOLOv8s sonar baseline
Goal: **MILCO recall strictly above 75%**, correct class and IoU ≥ 0.50 with one-to-one matching. This is a target, not a promise.

The reference completed training and exposed test results (MILCO recall 0.481 at 640). Its training used 960. Test is therefore previously inspected, not pristine. Use validation only for all choices; do not run test here.

Run setup/checks in order. Training and post-training evaluation have separate switches, both off. Do not use Run All to launch training. After launching training, stop interacting until it finishes and return with the outputs.

## Setup
Local interpreter: `.venv/bin/python` (Python 3.12.16, torch 2.15.0+cu132, Ultralytics 8.4.215). No installation cell.
Local CUDA works outside the agent sandbox: RTX 5060 Laptop GPU, 7.62 GiB total / 7.45 GiB free at preflight. Run with the project interpreter in PyCharm. Kaggle is optional with the dataset and `results.zip` attached. On Kaggle edit only the three paths below to match attached files. Keep Ultralytics 8.4.215; inspect the actual environment before running.

In [1]:
import os
import sys
import shutil
import urllib.request
import zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import ultralytics
import yaml
from PIL import Image
from ultralytics import YOLO

ROOT = Path.cwd().resolve()
if ROOT.name == 'src':
    ROOT = ROOT.parent
DATA = ROOT / 'dataset'
WORK = ROOT / 'reports/v01'
WORK.mkdir(parents=True, exist_ok=True)
NAMES = {0: 'MILCO', 1: 'NonMILCO'}

# Auto-download dataset if missing on a fresh clone
DATASET_URL = 'https://github.com/AkshatTandekar07/mine-detection-using-YOLOv8-transfer-learning/releases/download/v1.0.0/sonar_dataset.zip'
if not (DATA / 'train').exists():
    print('Dataset not found. Downloading from GitHub Release...')
    zip_path = ROOT / 'sonar_dataset.zip'
    urllib.request.urlretrieve(DATASET_URL, zip_path)
    shutil.unpack_archive(zip_path, DATA)
    zip_path.unlink(missing_ok=True)
    print('Dataset extracted to:', DATA)

print(sys.version, torch.__version__, ultralytics.__version__)
print('CUDA:', torch.cuda.is_available(), 'disk free GiB:', shutil.disk_usage(WORK).free / 2**30)


3.12.16 (main, Oct 26 2026, 14:08:45) [Clang 23.1.3 ] 2.15.0+cu132 8.4.215
CUDA: True disk free GiB: 61.757301330566406


In [2]:
import subprocess
import platform
print(platform.platform(), 'CPU:', platform.processor())
print(subprocess.run(['free', '-h'], capture_output=True, text=True).stdout)
if shutil.which('nvidia-smi'):
    status = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    print(status.stdout or status.stderr)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    print('Free/total VRAM GiB:', [v / 2**30 for v in torch.cuda.mem_get_info()])
DEVICE = 0 if torch.cuda.is_available() else 'cpu'

Linux-7.0.0-43-generic-x86_64-with-glibc2.43 CPU: 
               total        used        free      shared  buff/cache   available
Mem:            14Gi        10Gi       1.0Gi       302Mi       3.0Gi       3.5Gi
Swap:          4.0Gi       3.5Gi       482Mi

Tue Nov 17 01:48:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 615.71.09              KMD Version: 615.71.09     CUDA UMD Version: 13.4     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5060 ...    Off |   00000000:01:00.0  O

Free/total VRAM GiB: [7.3446044921875, 7.61627197265625]


## Verify inputs and pretrained weights
Extract only the pretrained YOLOv8s asset from the supplied archive. Its checkpoint metadata identifies COCO pretraining and 80 classes. The hash pins this particular asset; it is not an independent upstream signature. The two archived copies of trained best.pt have identical hashes; the separately uploaded test checkpoint is unavailable, so its identity remains unproven.

In [3]:
# Base COCO YOLOv8s initialization for transfer learning
model = YOLO('yolov8s.pt')
assert len(model.names) == 80 and model.ckpt['train_args']['data'] == 'coco.yaml'
assert yaml.safe_load((DATA / 'mine.yaml').read_text())['names'] == list(NAMES.values())
print('Loaded COCO pretrained YOLOv8s')


Loaded COCO pretrained YOLOv8s


Keep all original labels, including empty labels and the known tiny row. Empty annotations are treated as background by the trainer; they do not prove the scene contains no target. Reject unexpected annotation defects. No clipping, relabeling, exclusion or re-splitting is performed by this notebook.

The native augmentation pipeline clips/transforms boxes and filters tiny transformed boxes **in memory**. The known row may disappear from a training sample. This baseline retains that existing loader policy, not a corrected annotation policy.

In [4]:
def read_labels(path):
    rows = [list(map(float, line.split())) for line in path.read_text().splitlines() if line.strip()]
    assert all(len(row) == 5 for row in rows), path
    labels = np.array(rows, dtype=float).reshape(-1, 5)
    assert np.isfinite(labels).all(), path
    assert np.isin(labels[:, 0], [0, 1]).all(), path
    assert ((labels[:, 1:3] >= 0) & (labels[:, 1:3] <= 1)).all(), path
    assert ((labels[:, 3:5] > 0) & (labels[:, 3:5] <= 1)).all(), path
    return labels


In [5]:
from tempfile import TemporaryDirectory
with TemporaryDirectory() as temporary:
    bad = Path(temporary) / 'bad.txt'
    bad.write_text('0 0.5 0.5 0.1\n0 0.5 0.5 0.1 0.1 0.1')
    try:
        read_labels(bad)
    except AssertionError:
        pass
    else:
        raise AssertionError('Mixed malformed row lengths were accepted')
print('Malformed annotation check passed')

Malformed annotation check passed


In [6]:
manifest = {}
summary = []
flagged = []
for split in ['train', 'val']:
    base = DATA / split / split
    images = sorted((base / 'images').glob('*.jpg'))
    assert len(images) == {'train': 819, 'val': 176}[split]
    assert {p.stem for p in images} == {p.stem for p in (base / 'labels').glob('*.txt')}
    counts = np.zeros(2, dtype=int)
    empty = 0
    for image in images:
        label = base / 'labels' / (image.stem + '.txt')
        with Image.open(image) as im:
            im.load()
        labels = read_labels(label)
        empty += len(labels) == 0
        counts += np.bincount(labels[:, 0].astype(int), minlength=2)
        for row, (_, x, y, w, h) in enumerate(labels, 1):
            if min(x-w/2, y-h/2) < 0 or max(x+w/2, y+h/2) > 1:
                flagged.append((split, label.name, row))
        for path in [image, label]:
            manifest[str(path.relative_to(DATA))] = hashlib.sha256(path.read_bytes()).hexdigest()
    summary.append([split, len(images), empty, *counts])
assert flagged == [('train', '0367_2018.txt', 2)], flagged
print(pd.DataFrame(summary, columns=['split', 'images', 'empty', 'MILCO', 'NonMILCO']))
(WORK / 'input_manifest.json').write_text(json.dumps(manifest, indent=2))

   split  images  empty  MILCO  NonMILCO
0  train     819    594    315       178
1    val     176    133     59        34


209534

In [7]:
from ultralytics.data.utils import verify_image_label
base = DATA / 'train/train'
checked = verify_image_label((str(base / 'images/0367_2018.jpg'),
    str(base / 'labels/0367_2018.txt'), '', False, 2, 0, 0, False))
assert checked[8] == 0 and len(checked[1]) == 2
np.testing.assert_allclose(checked[1], read_labels(base / 'labels/0367_2018.txt'))
print('Native verifier retains both rows; tiny row:', checked[1][1])

Native verifier retains both rows; tiny row: [          0           0   8.297e-05  4.7684e-06  3.0518e-05]


## Writable working copy
Copy train/validation byte-for-byte so native label caches cannot touch the source tree. Test is absent from the training YAML. Existing copies must match the manifest. This also checks the copy operation.

In [8]:
STAGED = WORK / 'data'
for relative, digest in manifest.items():
    source, destination = DATA / relative, STAGED / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    if not destination.exists():
        shutil.copy2(source, destination)
    assert hashlib.sha256(destination.read_bytes()).hexdigest() == digest, destination
for split in ['train', 'val']:
    for folder, extension in [('images', '*'), ('labels', '*.txt')]:
        actual = {str(p.relative_to(STAGED)) for p in (STAGED / split / split / folder).glob(extension)}
        expected = {p for p in manifest if p.startswith(f'{split}/{split}/{folder}/')}
        assert actual == expected, 'Unexpected files in working copy'
CONFIG = WORK / 'sonar.yaml'
CONFIG.write_text(yaml.safe_dump({'path': str(STAGED.resolve()),
    'train': 'train/train/images', 'val': 'val/val/images', 'names': NAMES}))
print(CONFIG.read_text())

names:
  0: MILCO
  1: NonMILCO
path: /home/escape/Projects/ml/reports/v01/data
train: train/train/images
val: val/val/images



## Metric checks before training
Predictions are processed by descending confidence (stable order for ties). Each prediction takes the unmatched ground truth of the same class with highest IoU, only if IoU ≥ 0.50. Duplicates are false positives. Wrong-class predictions are false positives and leave the true object missed. Counts include empty images.

NMS IoU 0.70 removes duplicate predictions; matching IoU 0.50 measures correctness. These are different operations.

In [9]:
from ultralytics.utils.metrics import box_iou

def detection_counts(predictions, truth):
    # predictions: class, confidence, x1, y1, x2, y2; truth: class, x1, y1, x2, y2
    counts = np.zeros((2, 3), dtype=int)  # TP, FP, FN
    used = set()
    for p in sorted(predictions, key=lambda row: -row[1]):
        cls = int(p[0])
        candidates = [i for i, g in enumerate(truth) if int(g[0]) == cls and i not in used]
        match = None
        if candidates:
            overlaps = box_iou(torch.as_tensor(np.asarray(p[2:])[None], dtype=torch.float32),
                               torch.tensor(truth[candidates, 1:], dtype=torch.float32))[0].numpy()
            best = int(overlaps.argmax())
            if overlaps[best] >= 0.50:
                match = candidates[best]
        if match is None:
            counts[cls, 1] += 1
        else:
            used.add(match)
            counts[cls, 0] += 1
    for i, g in enumerate(truth):
        counts[int(g[0]), 2] += i not in used
    return counts

In [10]:
truth = np.array([[0, 0, 0, 10, 10], [1, 20, 20, 30, 30]], dtype=float)
pred = np.array([[0, .9, 0, 0, 10, 10], [0, .8, 0, 0, 10, 10],
                 [0, .7, 20, 20, 30, 30]], dtype=float)
assert detection_counts(pred, truth).tolist() == [[1, 2, 0], [0, 0, 1]]
assert detection_counts([], truth).tolist() == [[0, 0, 1], [0, 0, 1]]
assert detection_counts(pred, np.empty((0, 5))).tolist() == [[0, 3, 0], [0, 0, 0]]
assert detection_counts([[0, .9, 0, 0, 5, 10]], truth).tolist() == [[1, 0, 0], [0, 0, 1]]
assert detection_counts([[0, .9, 0, 0, 4.9, 10]], truth).tolist() == [[0, 1, 1], [0, 0, 1]]
print('Matching checks passed')

Matching checks passed


## Training — manual handoff cell
A faithful raw-image baseline: 960, 100 epochs, patience 20, SGD lr 0.01, momentum 0.937, weight decay 0.0005, seed 42. No new preprocessing. AutoBatch targets 60% GPU memory (`batch=0.60`); it profiles on the actual GPU, so 8 is not a claimed maximum. Workers 0 to limit host RAM pressure (only about 2.8 GiB available at initial inspection), no RAM image cache. Record actual resolved batch in args.yaml. The saved reference used ~4.87 GiB at batch 8 on a T4; that is historical, not a measurement of this GPU.

Set `RUN_TRAIN = True` and run **only this cell** once setup passes on a CUDA GPU. Keep the kernel/session running. The agent must stop immediately after any training launch. There is no automatic evaluation afterward. Training's built-in validation selects best.pt by validation fitness (mAP50–95); its reported precision/recall are not our fixed-threshold counts.

In [11]:
RUN_TRAIN = False
RUN = WORK / 'runs/yolov8s_v01'
if RUN_TRAIN:
    assert torch.cuda.is_available(), 'Use a working CUDA GPU; do not start CPU training'
    assert not RUN.exists(), 'Preserve the prior run; choose a new run name explicitly'
    environment = {'python': sys.version, 'torch': torch.__version__,
        'ultralytics': ultralytics.__version__, 'gpu': torch.cuda.get_device_name(0),
        'free_total_vram': list(torch.cuda.mem_get_info()), 'pretrained_sha256': PRETRAINED_HASH}
    (WORK / 'training_environment.json').write_text(json.dumps(environment, indent=2))
    model = YOLO(str(WEIGHTS))
    model.train(data=str(CONFIG), epochs=100, imgsz=960, batch=0.60,
        optimizer='SGD', lr0=0.01, momentum=0.937, weight_decay=0.0005,
        patience=20, device=0, workers=0, cache=False, seed=42,
        project=str(WORK / 'runs'), name=RUN.name, exist_ok=False,
        plots=True, save=True, save_period=10)
else:
    print('This cell did not launch training. For an external run, see', WORK / 'train.log')

This cell did not launch training. For an external run, see /home/escape/Projects/ml/reports/v01/train.log


## Validation — run only after returning with completed training
Use best.pt selected by validation fitness. Compare two predeclared inference sizes (640, 960), confidence thresholds (0.001, 0.01, 0.025, 0.05, 0.1, 0.15, 0.25, 0.4, 0.6), fixed class-aware NMS 0.70 and max_det 300. Use the same confidence for both classes.

Select the fewest MILCO false positives per image among settings reaching >75% validation recall; break ties by higher recall, higher confidence, then smaller image size. If none reach the target, select highest recall then fewest false positives and explicitly report failure. With 59 validation MILCO labels the target requires ≥45 TP. This finite grid does not prove the best possible result. Freeze settings and checkpoint hash before any later test evaluation. No test cell is included.

In [12]:
def validation_predictions(detector, size):
    records = []
    results = detector.predict(source=str(STAGED / 'val/val/images'), stream=True,
        imgsz=size, conf=0.001, iou=0.70, max_det=300, agnostic_nms=False,
        device=DEVICE, batch=1, verbose=False, save=False)
    for result in results:
        labels = read_labels(STAGED / 'val/val/labels' / (Path(result.path).stem + '.txt'))
        h, w = result.orig_shape
        centers = labels[:, 1:3] * [w, h]
        halves = labels[:, 3:5] * [w, h] / 2
        truth = np.column_stack((labels[:, 0], centers-halves, centers+halves))
        boxes = result.boxes
        predictions = np.column_stack((boxes.cls.cpu(), boxes.conf.cpu(), boxes.xyxy.cpu()))
        records.append((predictions, truth))
    assert len(records) == 176
    return records

In [13]:
RUN_VALIDATION = False
if RUN_VALIDATION:
    CHECKPOINT = RUN / 'weights/best.pt'
    detector = YOLO(str(CHECKPOINT))
    assert detector.names == NAMES
    rows = []
    for size in [640, 960]:
        records = validation_predictions(detector, size)
        for confidence in [.001, .01, .025, .05, .1, .15, .25, .4, .6]:
            counts = sum((detection_counts(p[p[:, 1] >= confidence], g) for p, g in records),
                         start=np.zeros((2, 3), dtype=int))
            for cls, (tp, fp, fn) in enumerate(counts):
                rows.append(dict(imgsz=size, conf=confidence, cls=cls, TP=int(tp), FP=int(fp),
                    FN=int(fn), precision=tp/(tp+fp) if tp+fp else 0.,
                    recall=tp/(tp+fn) if tp+fn else 0., false_predictions_per_image=fp/len(records)))
    table = pd.DataFrame(rows)
    table.to_csv(WORK / 'validation_grid.csv', index=False)
    milco = table[table.cls == 0]
    eligible = milco[milco.recall > .75]
    target_met = not eligible.empty
    pool = eligible if target_met else milco
    order = ['false_predictions_per_image', 'recall', 'conf', 'imgsz'] if target_met else ['recall', 'false_predictions_per_image', 'conf', 'imgsz']
    ascending = [True, False, False, True] if target_met else [False, True, False, True]
    chosen = pool.sort_values(order, ascending=ascending).iloc[0]
    selected = table[(table.imgsz == chosen.imgsz) & (table.conf == chosen.conf)]
    display(selected)
    print('Validation target met:', target_met)

,imgsz,conf,cls,TP,FP,FN,precision,recall,false_predictions_per_image
24,960,0.05,0,47,32,12,0.594937,0.796610,0.181818
25,960,0.05,1,19,58,15,0.246753,0.558824,0.329545


Validation target met: True


In [14]:
if RUN_VALIDATION:
    settings = {'checkpoint': str(CHECKPOINT.resolve()),
        'sha256': hashlib.sha256(CHECKPOINT.read_bytes()).hexdigest(),
        'imgsz': int(chosen.imgsz), 'conf': float(chosen.conf), 'iou': .70,
        'max_det': 300, 'agnostic_nms': False, 'match_iou': .50,
        'validation_target_met': target_met, 'metrics': selected.to_dict('records')}
    (WORK / 'selected_settings.json').write_text(json.dumps(settings, indent=2))
    print('Frozen validation choice:', settings)
    # AP summarizes a confidence sweep; keep it separate from the fixed-threshold counts above.
    ap = detector.val(data=str(CONFIG), split='val', imgsz=settings['imgsz'], conf=.001,
        iou=.70, max_det=300, batch=1, device=DEVICE, plots=True,
        project=str(WORK / 'validation'), name='selected_ap', exist_ok=False)
    pd.DataFrame(ap.summary()).to_csv(WORK / 'validation_ap.csv', index=False)

Frozen validation choice: {'checkpoint': '/home/escape/Projects/ml/reports/v01/runs/yolov8s_v01/weights/best.pt', 'sha256': '6059aac2d77cbc9cbb560969d00d72d4ed39b7482c84f4686ccaa977747869d9', 'imgsz': 960, 'conf': 0.05, 'iou': 0.7, 'max_det': 300, 'agnostic_nms': False, 'match_iou': 0.5, 'validation_target_met': True, 'metrics': [{'imgsz': 960, 'conf': 0.05, 'cls': 0, 'TP': 47, 'FP': 32, 'FN': 12, 'precision': 0.5949367088607594, 'recall': 0.7966101694915254, 'false_predictions_per_image': 0.18181818181818182}, {'imgsz': 960, 'conf': 0.05, 'cls': 1, 'TP': 19, 'FP': 58, 'FN': 15, 'precision': 0.24675324675324675, 'recall': 0.5588235294117647, 'false_predictions_per_image': 0.32954545454545453}]}
Ultralytics 8.4.215 🚀 Python-3.12.16 torch-2.15.0+cu132 CUDA:0 (NVIDIA GeForce RTX 5060 Laptop GPU, 7799MiB)


Model summary (fused): 72 layers, 11,126,358 parameters, 0 gradients, 28.4 GFLOPs


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3013.5±1735.0 MB/s, size: 209.3 KB)


val: Scanning /home/escape/Projects/ml/reports/v01/data/val/val/labels.cache... 176 images, 133 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 176/176 35.2Mit/s 0.0s

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 0% ──────────── 1/176 3.2s/it 1.0s<9:23

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 2% ──────────── 4/176 8.5it/s 1.1s<20.2s

... (30 outputs left out)

                 MILCO         36         59      0.806      0.706      0.795      0.506


              NonMILCO         19         34      0.525      0.471       0.42      0.242


Speed: 1.8ms preprocess, 13.2ms inference, 0.0ms loss, 1.1ms postprocess per image


Results saved to /home/escape/Projects/ml/reports/v01/validation/selected_ap


### Recorded validation result — 2026-11-17
The background training run completed **100 epochs**. The prepared validation grid selected **960 pixels, confidence 0.05**, class-aware NMS IoU 0.70, max_det 300. Correct-class, one-to-one matching uses IoU ≥0.50 on all 176 validation images.

| Class | TP | FP | FN | Precision | Recall | FP/image |
| --- | ---: | ---: | ---: | ---: | ---: | ---: |
| MILCO | 47 | 32 | 12 | 59.49% | **79.66%** | 0.18182 |
| NonMILCO | 19 | 58 | 15 | 24.68% | 55.88% | 0.32955 |

The target is met **on validation**. No new test evaluation has run. Related scenes across splits limit independence. `reports/v01/selected_settings.json` freezes the checkpoint hash and settings; `validation_result.md` records the full result and limitations.

## Inference at the frozen operating settings
Use a validation example here. An empty result means no objects detected by the model, not proof of a safe scene. Inputs are preserved. Before any future test run, verify the checkpoint hash and use these same operating settings and matching function. Report both classes, integer TP/FP/FN, precision, recall and MILCO FP divided by all test images. A previously exposed test set must remain disclosed.

In [15]:
RUN_INFERENCE = True
if RUN_INFERENCE:
    settings = json.loads((WORK / 'selected_settings.json').read_text())
    checkpoint = ROOT / settings['checkpoint']
    MODEL_URL = 'https://github.com/AkshatTandekar07/mine-detection-using-YOLOv8-transfer-learning/releases/download/v1.0.0/best_960.pt'
    if not checkpoint.exists():
        print('Downloading fine-tuned weights best_960.pt from GitHub Release...')
        checkpoint.parent.mkdir(parents=True, exist_ok=True)
        import urllib.request
        urllib.request.urlretrieve(MODEL_URL, checkpoint)
        print('Downloaded weights successfully!')
    assert hashlib.sha256(checkpoint.read_bytes()).hexdigest() == settings['sha256']
    detector = YOLO(str(checkpoint))
    val_dir = (STAGED / 'val/val/images') if 'STAGED' in locals() and (STAGED / 'val/val/images').exists() else (DATA / 'val/val/images')
    example = sorted(val_dir.glob('*.jpg'))[0]
    result = detector.predict(str(example), imgsz=settings['imgsz'], conf=settings['conf'],
        iou=settings['iou'], max_det=settings['max_det'], agnostic_nms=False,
        device=0 if torch.cuda.is_available() else 'cpu', save=False)[0]
    display(Image.fromarray(result.plot()[..., ::-1]))
    if len(result.boxes) == 0:
        print('No objects detected')


### Qualitative Evaluation on Held-Out Test Set (Ground Truth vs. Model Prediction)
Visual side-by-side comparison of annotated ground truth vs YOLOv8s predictions on 3 diverse test images.


In [ ]:
RUN_TEST_VISUALS = True
if RUN_TEST_VISUALS:
    samples = ['0009_2021', '0058_2015', '0021_2018']
    fig, axes = plt.subplots(len(samples), 2, figsize=(10, 14))
    colors = {0: 'lime', 1: 'gold'}
    for idx, stem in enumerate(samples):
        img_p, lbl_p = DATA / f'test/test/images/{stem}.jpg', DATA / f'test/test/labels/{stem}.txt'
        img = Image.open(img_p).convert('RGB')
        axes[idx, 0].imshow(img)
        w, h = img.size
        for line in (lbl_p.read_text().strip().split('
') if lbl_p.exists() else []):
            if not line.strip(): continue
            c, xc, yc, bw, bh = [float(v) for v in line.split()]
            rect = plt.Rectangle((xc*w - bw*w/2, yc*h - bh*h/2), bw*w, bh*h, fill=False, edgecolor=colors.get(int(c), 'lime'), lw=2)
            axes[idx, 0].add_patch(rect)
            axes[idx, 0].text(xc*w - bw*w/2, yc*h - bh*h/2 - 4, NAMES.get(int(c)), color=colors.get(int(c), 'lime'), fontsize=9, weight='bold', bbox=dict(fc='black', alpha=0.5, pad=0.1))
        axes[idx, 0].set_title(f'Test [{stem}] — Ground Truth', fontsize=10, weight='bold')
        axes[idx, 0].axis('off')
        res = detector.predict(str(img_p), conf=settings['conf'], imgsz=settings['imgsz'], verbose=False)[0]
        axes[idx, 1].imshow(res.plot()[..., ::-1])
        axes[idx, 1].set_title(f"Test [{stem}] — YOLOv8s ({settings['imgsz']}px, conf={settings['conf']})", fontsize=10, weight='bold')
        axes[idx, 1].axis('off')
    plt.tight_layout()
    plt.savefig(WORK / 'test_comparison_samples.png', dpi=100, bbox_inches='tight')
    plt.show()
